# Test with a real data

In [ ]:
import xarray as xr

In [ ]:
#datadir = "/global/cfs/cdirs/m4581/lee1043/DATA/CMIP6/HighResMIP/ECMWF-IFS-LR/hist-1950/6hrPlevPt"
#datadir = "/global/cfs/cdirs/m4581/lee1043/DATA/CMIP6/HighResMIP/ECMWF-IFS-LR/highresSST-present/6hrPlevPt"
#datadir = "/home/lee1043/work/DATA/HighResMIP/ECMWF-IFS-LR/hist-1950/6hrPlevPt"
datadir = "/home/lee1043/work/DATA/HighResMIP/ECMWF-IFS-LR/highresSST-present/6hrPlevPt"

In [ ]:
import glob

data_list = sorted(glob.glob(f"{datadir}/*/*ECMWF-IFS-LR*_2014*.nc"))
data_list

In [ ]:
target_months = ('201403', '201406', '201409', '201412')

data_list = [
    data_path
    for data_path in data_list
    if any(month in data_path for month in target_months)
]

In [ ]:
data_list

In [ ]:
ds = xr.open_mfdataset(data_list)

In [ ]:
ds

In [ ]:
from pcmdi_metrics.effective_resolution import compute_effective_resolution

In [ ]:
%%time

metrics, diagnostics = compute_effective_resolution(
    ds,
    uvar="ua",
    vvar="va",
    levels=(250.0, 500.0),
    model="ECMWF-IFS-LR",
    member="r1i1p1f1",
)

result = metrics["ECMWF-IFS-LR"]["r1i1p1f1"]
result

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Plot the kinetic energy spectra
spectra = diagnostics["spectra"]
colors = {"div": "C0", "rot_250": "C1", "rot_500": "C2"}
labels = {"div": "Divergent 250 hPa", "rot_250": "Rotational 250 hPa", 
          "rot_500": "Rotational 500 hPa"}

# Plot divergent at 250 hPa
spec_250 = spectra[250.0]
wavenumber = spec_250["wavenumber"].values
ke_div = spec_250["ke_div"].values
ax.loglog(wavenumber, ke_div, color=colors["div"], linewidth=2, label=labels["div"])

# Plot rotational at 250 hPa
ke_rot_250 = spec_250["ke_rot"].values
ax.loglog(wavenumber, ke_rot_250, color=colors["rot_250"], linewidth=2, label=labels["rot_250"])

# Plot rotational at 500 hPa
spec_500 = spectra[500.0]
ke_rot_500 = spec_500["ke_rot"].values
ax.loglog(wavenumber, ke_rot_500, color=colors["rot_500"], linewidth=2, label=labels["rot_500"])

# Add reference lines for k^-3 and k^-5/3
k_ref = np.logspace(0, 3, 100)  # wavenumbers from 1 to 1000
# Normalize reference lines to intersect the data appropriately
ke_ref_k3 = 1e1 * k_ref**(-3)
ke_ref_k53 = 1e1 * k_ref**(-5/3)

ax.loglog(k_ref, ke_ref_k3, 'k-', linewidth=1.5, alpha=0.6, label=r'$k^{-3}$')
ax.loglog(k_ref, ke_ref_k53, 'k--', linewidth=1.5, alpha=0.6, label=r'$k^{-5/3}$')

ax.set_xlim(1, 1000)
ax.set_ylim(1e-7, 1e2)
ax.set_xlabel("Total horizontal wavenumber", fontsize=11)
ax.set_ylabel(r"Kinetic energy per unit mass (m$^2$ s$^{-2}$)", fontsize=11)
ax.legend(loc="upper right", frameon=True, framealpha=0.9)
ax.grid(True, which="both", alpha=0.3, linestyle=":")
ax.set_title("Kinetic energy spectra — ECMWF-IFS-LR", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
from pcmdi_metrics.effective_resolution.lib import (
    plot_spectra_and_slope,
)

In [ ]:
fig = plot_spectra_and_slope(diagnostics, result, title="ECMWF-IFS-LR")
plt.show()

In [ ]:
# Match paper Figure 1 axis ranges
fig = plot_spectra_and_slope(
    diagnostics, 
    result,
    xlim=(13, 240),          # Wavenumber range
    spec_ylim=(1e0, 1e2),    # Compensated spectrum y-range  
    slope_ylim=(1, 5)        # Slope panel y-range
)

In [ ]:
metrics